<a href="https://colab.research.google.com/github/SANGHATI23/sam-brats-robustness-audit/blob/main/00_MedReasoner_SAM_Data_Audit_and_Benchmark_Setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Med-Reasoner / NeurIPS 2026 — SAM Failure-Reasoning Study
## Notebook 00: Data Audit, Recovery, and Benchmark Setup

**Purpose:** Do **not** rerun the old SAM study yet. First recover what already exists.

This notebook will:

1. Mount Google Drive.
2. Create a new, permanent project folder in Drive.
3. Clone/update the existing public SAM robustness GitHub repository.
4. Search Drive for likely old **BraTS / KiTS / MoNuSeg / SAM** folders and files.
5. Parse the old GitHub notebooks to recover hard-coded path hints.
6. Inventory existing CSV results and identify per-case Dice tables.
7. Reconstruct a master failure-label table using the original paper thresholds:
   - Primary SAM failure: **Dice < 0.50**
   - Severe SAM failure: **Dice < 0.10**
8. Create a reproducible candidate benchmark manifest for:
   - Brain MRI / BraTS
   - Kidney CT / KiTS23
   - Histopathology / MoNuSeg
9. Report what is already available and what, if anything, must be regenerated.

### Runtime
For this notebook alone, CPU/T4 is sufficient.  
For the full project, use **GPU → A100 if available** so later local VLM inference and any SAM regeneration can use the same runtime.

### Important
This notebook does **not** modify the accepted IEEE-ICHI study. It creates a separate Med-Reasoner project workspace.

In [1]:
# CELL 1 — Environment / GPU check
import os, sys, json, re, subprocess, platform, shutil, time
from pathlib import Path
import pandas as pd
import numpy as np

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        props = torch.cuda.get_device_properties(0)
        print(f"GPU memory: {props.total_memory / 1024**3:.1f} GB")
    else:
        print("GPU not available. That is OK for Notebook 00.")
except Exception as e:
    print("Torch check skipped:", e)

Python: 3.12.13
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
GPU memory: 79.3 GB


In [2]:
# CELL 2 — Mount Google Drive
# If Colab asks for permission, approve it.
DRIVE_MOUNT = Path("/content/drive")

try:
    from google.colab import drive
    if not (DRIVE_MOUNT / "MyDrive").exists():
        drive.mount(str(DRIVE_MOUNT), force_remount=False)
    else:
        print("Drive already mounted.")
except Exception as e:
    print("Drive mount failed:", repr(e))
    print("You can rerun this cell. The GitHub audit can still work without Drive.")

MYDRIVE = DRIVE_MOUNT / "MyDrive"
print("MyDrive available:", MYDRIVE.exists())

Mounted at /content/drive
MyDrive available: True


In [3]:
# CELL 3 — Create a NEW permanent project folder
# Everything generated by the new Med-Reasoner work will be kept here.
if MYDRIVE.exists():
    PROJECT_ROOT = MYDRIVE / "MedReasoner_SAM_2026"
else:
    PROJECT_ROOT = Path("/content/MedReasoner_SAM_2026")

DIRS = {
    "audit": PROJECT_ROOT / "00_setup_audit",
    "manifests": PROJECT_ROOT / "01_manifests",
    "assets": PROJECT_ROOT / "02_case_assets",
    "vlm_outputs": PROJECT_ROOT / "03_vlm_outputs",
    "counterfactuals": PROJECT_ROOT / "04_counterfactual_masks",
    "evaluation": PROJECT_ROOT / "05_evaluation",
    "figures": PROJECT_ROOT / "06_figures",
    "logs": PROJECT_ROOT / "logs",
}

for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

print("NEW project root:", PROJECT_ROOT)
for k, p in DIRS.items():
    print(f"  {k:16s} -> {p}")

NEW project root: /content/drive/MyDrive/MedReasoner_SAM_2026
  audit            -> /content/drive/MyDrive/MedReasoner_SAM_2026/00_setup_audit
  manifests        -> /content/drive/MyDrive/MedReasoner_SAM_2026/01_manifests
  assets           -> /content/drive/MyDrive/MedReasoner_SAM_2026/02_case_assets
  vlm_outputs      -> /content/drive/MyDrive/MedReasoner_SAM_2026/03_vlm_outputs
  counterfactuals  -> /content/drive/MyDrive/MedReasoner_SAM_2026/04_counterfactual_masks
  evaluation       -> /content/drive/MyDrive/MedReasoner_SAM_2026/05_evaluation
  figures          -> /content/drive/MyDrive/MedReasoner_SAM_2026/06_figures
  logs             -> /content/drive/MyDrive/MedReasoner_SAM_2026/logs


In [4]:
# CELL 4 — Clone or update your existing SAM robustness GitHub repository
REPO_URL = "https://github.com/SANGHATI23/sam-brats-robustness-audit.git"
REPO_DIR = Path("/content/sam-brats-robustness-audit")

if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    print("Repository already cloned. Pulling latest main...")
    result = subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
        text=True, capture_output=True
    )
    print(result.stdout[-2000:])
    if result.returncode != 0:
        print("git pull warning:", result.stderr[-2000:])
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

print("Repo:", REPO_DIR)
print("Exists:", REPO_DIR.exists())

files = [p for p in REPO_DIR.rglob("*") if p.is_file()]
print("Total repository files:", len(files))
print("Notebooks:", sum(p.suffix == ".ipynb" for p in files))
print("CSVs:", sum(p.suffix.lower() == ".csv" for p in files))
print("PNG/JPG images:", sum(p.suffix.lower() in {".png",".jpg",".jpeg"} for p in files))

Repo: /content/sam-brats-robustness-audit
Exists: True
Total repository files: 212
Notebooks: 5
CSVs: 15
PNG/JPG images: 161


In [5]:
# CELL 5 — Targeted Drive search for old project/data folders
SEARCH_TOKENS = [
    "brats", "kits", "kidney", "monuseg", "histopath",
    "sam-brats", "sam_brats", "robustness"
]

def targeted_find(root: Path, token: str, item_type="any", maxdepth=7, timeout=90):
    if not root.exists():
        return []
    cmd = ["find", str(root), "-maxdepth", str(maxdepth)]
    if item_type == "dir":
        cmd += ["-type", "d"]
    elif item_type == "file":
        cmd += ["-type", "f"]
    cmd += ["-iname", f"*{token}*"]
    try:
        r = subprocess.run(cmd, text=True, capture_output=True, timeout=timeout)
        return [x.strip() for x in r.stdout.splitlines() if x.strip()]
    except subprocess.TimeoutExpired:
        print(f"Search timed out for token: {token}")
        return []

drive_hits = []
if MYDRIVE.exists():
    for tok in SEARCH_TOKENS:
        hits = targeted_find(MYDRIVE, tok, "any")
        for h in hits[:300]:
            drive_hits.append({"token": tok, "path": h, "exists": Path(h).exists()})
else:
    print("Drive not mounted; skipping Drive search.")

drive_hits_df = pd.DataFrame(drive_hits).drop_duplicates() if drive_hits else pd.DataFrame(
    columns=["token","path","exists"]
)

display(drive_hits_df.head(100))
print("Total matching Drive paths:", len(drive_hits_df))

drive_hits_path = DIRS["audit"] / "drive_search_hits.csv"
drive_hits_df.to_csv(drive_hits_path, index=False)
print("Saved:", drive_hits_path)

Search timed out for token: brats
Search timed out for token: kits
Search timed out for token: kidney


,token,path,exists
0,histopath,/content/drive/MyDrive/Colab Notebooks/Histopa...,True
1,robustness,/content/drive/MyDrive/Colab Notebooks/Copy of...,True
2,robustness,/content/drive/MyDrive/Colab Notebooks/sam_spl...,True
3,robustness,/content/drive/MyDrive/Colab Notebooks/05_Neur...,True
4,robustness,/content/drive/MyDrive/Robustness Evaluation o...,True
5,robustness,/content/drive/MyDrive/Robustness Evaluation o...,True
6,robustness,/content/drive/MyDrive/neurofhir-qc/backend/ap...,True
7,robustness,/content/drive/MyDrive/neurofhir-qc/evaluation...,True
8,robustness,/content/drive/MyDrive/neurofhir-qc/evaluation...,True
9,robustness,/content/drive/MyDrive/neurofhir-qc/evaluation...,True


Total matching Drive paths: 36
Saved: /content/drive/MyDrive/MedReasoner_SAM_2026/00_setup_audit/drive_search_hits.csv


In [6]:
# CELL 6 — Recover path hints embedded in your OLD GitHub notebooks
# This often finds dataset names/locations even if you no longer remember them.
PATH_PATTERNS = [
    r'(/content/drive/MyDrive/[^"\'\n]+)',
    r'(/content/[^"\'\n]+)',
]

DATASET_WORDS = re.compile(
    r"(brats|kits|kidney|monuseg|histopath|spleen|liver|sam|mask|nii|nifti)",
    flags=re.I
)

path_hints = []

for nb_path in sorted(REPO_DIR.rglob("*.ipynb")):
    try:
        obj = json.loads(nb_path.read_text(errors="ignore"))
    except Exception:
        continue

    for cell_idx, cell in enumerate(obj.get("cells", [])):
        src = "".join(cell.get("source", []))
        if not DATASET_WORDS.search(src):
            continue

        found = set()
        for pat in PATH_PATTERNS:
            found.update(re.findall(pat, src))

        # Also retain concise source lines that look path/data related.
        lines = [
            line.strip() for line in src.splitlines()
            if DATASET_WORDS.search(line)
            and len(line.strip()) <= 300
            and not line.strip().startswith("#")
        ]

        for p in sorted(found):
            path_hints.append({
                "notebook": nb_path.name,
                "cell": cell_idx,
                "type": "absolute_path",
                "hint": p
            })

        for line in lines[:40]:
            path_hints.append({
                "notebook": nb_path.name,
                "cell": cell_idx,
                "type": "source_line",
                "hint": line
            })

path_hints_df = pd.DataFrame(path_hints).drop_duplicates() if path_hints else pd.DataFrame(
    columns=["notebook","cell","type","hint"]
)

display(path_hints_df.head(150))
print("Recovered hints:", len(path_hints_df))

hints_path = DIRS["audit"] / "old_notebook_path_hints.csv"
path_hints_df.to_csv(hints_path, index=False)
print("Saved:", hints_path)

,notebook,cell,type,hint
0,Copy_of_01_brain_brats_.ipynb,1,source_line,"gt_folder = ""gt_masks"""
1,Copy_of_01_brain_brats_.ipynb,1,source_line,"pred_folder = ""pred_masks"""
2,Copy_of_01_brain_brats_.ipynb,7,source_line,!wget -q --show-progress -O checkpoints/sam_vi...
3,Copy_of_01_brain_brats_.ipynb,7,source_line,https://dl.fbaipublicfiles.com/segment_anythin...
4,Copy_of_01_brain_brats_.ipynb,7,source_line,!ls -lh checkpoints/sam_vit_b_01ec64.pth
...,...,...,...,...
145,Copy_of_01_brain_brats_.ipynb,20,source_line,sam.to(device=device)
146,Copy_of_01_brain_brats_.ipynb,20,source_line,predictor = SamPredictor(sam)
147,Copy_of_01_brain_brats_.ipynb,20,source_line,"def bbox_from_mask(mask, pad=10):"
148,Copy_of_01_brain_brats_.ipynb,20,source_line,"ys, xs = np.where(mask > 0)"


Recovered hints: 632
Saved: /content/drive/MyDrive/MedReasoner_SAM_2026/00_setup_audit/old_notebook_path_hints.csv


In [7]:
# CELL 7 — Inventory every CSV in the GitHub repo
csv_inventory = []

for csv_path in sorted(REPO_DIR.rglob("*.csv")):
    try:
        df = pd.read_csv(csv_path)
        csv_inventory.append({
            "file": str(csv_path.relative_to(REPO_DIR)),
            "rows": len(df),
            "columns": " | ".join(map(str, df.columns)),
            "has_dice": any(str(c).lower() == "dice" for c in df.columns),
            "has_iou": any(str(c).lower() == "iou" for c in df.columns),
        })
    except Exception as e:
        csv_inventory.append({
            "file": str(csv_path.relative_to(REPO_DIR)),
            "rows": None,
            "columns": f"READ ERROR: {e}",
            "has_dice": False,
            "has_iou": False,
        })

csv_inventory_df = pd.DataFrame(csv_inventory)
display(csv_inventory_df)

csv_inv_path = DIRS["audit"] / "github_csv_inventory.csv"
csv_inventory_df.to_csv(csv_inv_path, index=False)
print("Saved:", csv_inv_path)

,file,rows,columns,has_dice,has_iou
0,brain_prompt_variants.csv,300,id | dice_score_select | dice_oracle_select | ...,False,False
1,brain_results.csv,300,id | case_id | z | sam_score | dice | iou | se...,True,True
2,brain_robustness_summary.csv,9,family | level | mean_dice | std_dice | min_di...,False,False
3,combined_summary.csv,5,experiment | mean_dice | std_dice | min_dice |...,False,False
4,kits23_tumor_robustness_summary_table.csv,11,condition | mean_dice | mean_iou | mean_delta_...,False,False
5,kits23_tumor_sam_boxprompt_metrics.csv,1987,file | dice | iou,True,True
6,kits23_tumor_sam_robustness_results.csv,21857,file | condition | dice | iou,True,True
7,liver_results.csv,300,id | case_id | z | sam_score | dice | iou | se...,True,True
8,liver_robustness_summary.csv,25,family | level | mean_dice | std_dice | min_di...,False,False
9,monuseg_robustness_summary_table.csv,11,condition | mean_dice | mean_iou | mean_delta_...,False,False


Saved: /content/drive/MyDrive/MedReasoner_SAM_2026/00_setup_audit/github_csv_inventory.csv


In [14]:
# CELL 8 — FIXED: Build clean baseline master table
# Uses ONLY the three clean per-case baseline tables needed for Med-Reasoner.

import re
import numpy as np
import pandas as pd

BASELINE_TABLES = {
    "brain_results.csv": "brain_mri",
    "kits23_tumor_sam_boxprompt_metrics.csv": "kidney_ct",
    "monuseg_sam_boxprompt_metrics.csv": "histopathology",
}

def find_col(cols, candidates):
    low = {str(c).lower(): c for c in cols}
    for c in candidates:
        if c in low:
            return low[c]
    return None

def extract_z(text):
    m = re.search(r"_z(\d+)", str(text), flags=re.I)
    return int(m.group(1)) if m else np.nan

def extract_case(text):
    s = str(text)
    m = re.search(r"(case_\d+)", s, flags=re.I)
    if m:
        return m.group(1)

    # BraTS-like identifier
    m = re.search(r"(BRATS[_-]?\d+)", s, flags=re.I)
    if m:
        return m.group(1)

    return s

master_parts = []

for filename, domain in BASELINE_TABLES.items():
    csv_path = REPO_DIR / filename

    if not csv_path.exists():
        print("MISSING:", csv_path)
        continue

    df = pd.read_csv(csv_path)

    dice_col = find_col(df.columns, ["dice", "dsc", "dice_score"])
    iou_col = find_col(df.columns, ["iou", "jaccard"])
    id_col = find_col(
        df.columns,
        ["id", "file", "filename", "image", "image_id", "sample_id"]
    )
    case_col = find_col(df.columns, ["case_id", "case", "subject", "patient"])
    z_col = find_col(df.columns, ["z", "slice_idx", "slice_index"])
    score_col = find_col(df.columns, ["sam_score", "score", "predicted_iou"])

    if dice_col is None:
        print("NO DICE COLUMN:", filename)
        continue

    # IMPORTANT FIX: initialize dataframe WITH the correct number of rows.
    tmp = pd.DataFrame(index=np.arange(len(df)))

    tmp["source_csv"] = filename
    tmp["domain"] = domain
    tmp["row_index"] = np.arange(len(df))

    if id_col is not None:
        tmp["record_id"] = df[id_col].astype(str).values
    else:
        tmp["record_id"] = [
            f"{domain}_{i:05d}" for i in range(len(df))
        ]

    if case_col is not None:
        tmp["case_id"] = df[case_col].astype(str).values
    else:
        tmp["case_id"] = tmp["record_id"].map(extract_case)

    if z_col is not None:
        tmp["z"] = pd.to_numeric(df[z_col], errors="coerce").values
    else:
        tmp["z"] = tmp["record_id"].map(extract_z)

    tmp["dice"] = pd.to_numeric(df[dice_col], errors="coerce").values

    if iou_col is not None:
        tmp["iou"] = pd.to_numeric(df[iou_col], errors="coerce").values
    else:
        tmp["iou"] = np.nan

    if score_col is not None:
        tmp["sam_score"] = pd.to_numeric(
            df[score_col], errors="coerce"
        ).values
    else:
        tmp["sam_score"] = np.nan

    tmp = tmp[tmp["dice"].between(0, 1, inclusive="both")].copy()

    # Original ICHI definitions
    tmp["sam_failure_primary"] = tmp["dice"] < 0.50
    tmp["sam_failure_severe"] = tmp["dice"] < 0.10

    # Sampling strata for the NEW VLM benchmark
    tmp["sampling_stratum"] = pd.cut(
        tmp["dice"],
        bins=[-0.001, 0.10, 0.50, 0.75, 1.001],
        labels=[
            "severe_<0.10",
            "failure_0.10-0.49",
            "borderline_0.50-0.74",
            "strong_>=0.75"
        ],
        right=False
    ).astype(str)

    master_parts.append(tmp)

master = pd.concat(master_parts, ignore_index=True)

master_path = DIRS["manifests"] / "master_sam_case_metrics.csv"
master.to_csv(master_path, index=False)

print("=" * 70)
print("FIXED MASTER TABLE")
print("=" * 70)
print("Total clean baseline cases:", len(master))
print()
print(master.groupby("domain").size())
print()

summary = (
    master.groupby("domain")
    .agg(
        n=("dice", "size"),
        mean_dice=("dice", "mean"),
        failure_rate=("sam_failure_primary", "mean"),
        severe_failure_rate=("sam_failure_severe", "mean"),
    )
)

summary["failure_rate_pct"] = summary["failure_rate"] * 100
summary["severe_failure_rate_pct"] = summary["severe_failure_rate"] * 100

display(summary)
display(master.head())

print("\nSaved:", master_path)

FIXED MASTER TABLE
Total clean baseline cases: 2338

domain
brain_mri          300
histopathology      51
kidney_ct         1987
dtype: int64



,n,mean_dice,failure_rate,severe_failure_rate,failure_rate_pct,severe_failure_rate_pct
domain,,,,,,
brain_mri,300,0.514879,0.393333,0.063333,39.333333,6.333333
histopathology,51,0.395921,0.784314,0.000000,78.431373,0.000000
kidney_ct,1987,0.837443,0.034726,0.007549,3.472572,0.754907


,source_csv,domain,row_index,record_id,case_id,z,dice,iou,sam_score,sam_failure_primary,sam_failure_severe,sampling_stratum
0,brain_results.csv,brain_mri,0,BRATS_482_z117,BRATS_482,117.0,0.756764,0.608705,0.963692,False,False,strong_>=0.75
1,brain_results.csv,brain_mri,1,BRATS_482_z094,BRATS_482,94.0,0.591879,0.420332,0.941994,False,False,borderline_0.50-0.74
2,brain_results.csv,brain_mri,2,BRATS_482_z079,BRATS_482,79.0,0.631616,0.461578,0.936099,False,False,borderline_0.50-0.74
3,brain_results.csv,brain_mri,3,BRATS_482_z066,BRATS_482,66.0,0.093730,0.049169,0.921791,True,True,severe_<0.10
4,brain_results.csv,brain_mri,4,BRATS_482_z092,BRATS_482,92.0,0.542339,0.372061,0.951686,False,False,borderline_0.50-0.74



Saved: /content/drive/MyDrive/MedReasoner_SAM_2026/01_manifests/master_sam_case_metrics.csv


In [9]:
# CELL 9 — Verify recovered metrics against the published study pattern
if len(master):
    summary = (
        master.groupby(["domain"], dropna=False)
        .agg(
            n=("dice","size"),
            mean_dice=("dice","mean"),
            median_dice=("dice","median"),
            failure_rate=("sam_failure_primary","mean"),
            severe_failure_rate=("sam_failure_severe","mean"),
        )
        .reset_index()
    )
    summary["failure_rate_pct"] = 100 * summary["failure_rate"]
    summary["severe_failure_rate_pct"] = 100 * summary["severe_failure_rate"]

    display(summary)

    strata = (
        master.groupby(["domain","sampling_stratum"])
        .size().reset_index(name="n")
        .sort_values(["domain","sampling_stratum"])
    )
    display(strata)

    summary.to_csv(DIRS["audit"] / "recovered_metric_summary.csv", index=False)
    strata.to_csv(DIRS["audit"] / "recovered_sampling_strata.csv", index=False)
else:
    print("No reusable per-case metrics were recovered from repository CSVs.")

,domain,n,mean_dice,median_dice,failure_rate,severe_failure_rate,failure_rate_pct,severe_failure_rate_pct
0,NaN,32677,0.857413,0.895911,0.032316,0.00303,3.231631,0.302965


,domain,sampling_stratum,n


## Build the Med-Reasoner candidate set

The **primary scientific label remains Dice < 0.50**, exactly as in the existing robustness study.  
The four Dice intervals below are only **sampling strata** to ensure the VLM benchmark sees catastrophic, ordinary-failure, borderline, and strong cases.

The default target is up to **25 cases per stratum per domain**. If a domain does not contain enough cases in a stratum, the notebook uses all available cases rather than fabricating balance.

In [17]:
# CELL 10 — Reproducible candidate benchmark manifest
TARGET_DOMAINS = ["brain_mri", "kidney_ct", "histopathology"]
PER_STRATUM = 25
SEED = 2026

rng = np.random.default_rng(SEED)
selected_parts = []

if len(master):
    for domain in TARGET_DOMAINS:
        ddf = master[master["domain"] == domain].copy()

        # If inference from filenames missed a table, we do not silently relabel it.
        if ddf.empty:
            print(f"WARNING: no per-case metric table automatically identified for {domain}")
            continue

        for stratum in [
            "severe_<0.10",
            "failure_0.10-0.49",
            "borderline_0.50-0.74",
            "strong_>=0.75"
        ]:
            sdf = ddf[ddf["sampling_stratum"] == stratum].copy()
            n_take = min(PER_STRATUM, len(sdf))
            if n_take:
                idx = rng.choice(sdf.index.to_numpy(), size=n_take, replace=False)
                selected_parts.append(sdf.loc[idx])

candidate = (
    pd.concat(selected_parts, ignore_index=True)
    if selected_parts else
    pd.DataFrame(columns=master.columns)
)

if len(candidate):
    candidate = candidate.sort_values(
        ["domain","sampling_stratum","dice","record_id"]
    ).reset_index(drop=True)
    candidate.insert(0, "benchmark_id", [f"MR_{i:04d}" for i in range(1, len(candidate)+1)])

display(candidate)
print("Candidate benchmark rows:", len(candidate))

if len(candidate):
    print("\nBy domain / stratum:")
    display(candidate.groupby(["domain","sampling_stratum"]).size().reset_index(name="n"))

candidate_path = DIRS["manifests"] / "candidate_medreasoner_benchmark.csv"
candidate.to_csv(candidate_path, index=False)
print("Saved:", candidate_path)

,benchmark_id,source_csv,domain,row_index,record_id,case_id,z,dice,iou,sam_score,sam_failure_primary,sam_failure_severe,sampling_stratum
0,MR_0001,brain_results.csv,brain_mri,45,BRATS_371_z075,BRATS_371,75.0,0.506769,0.339378,0.954224,False,False,borderline_0.50-0.74
1,MR_0002,brain_results.csv,brain_mri,297,BRATS_010_z091,BRATS_010,91.0,0.507427,0.339968,0.952594,False,False,borderline_0.50-0.74
2,MR_0003,brain_results.csv,brain_mri,196,BRATS_152_z091,BRATS_152,91.0,0.508024,0.340504,0.978911,False,False,borderline_0.50-0.74
3,MR_0004,brain_results.csv,brain_mri,137,BRATS_476_z096,BRATS_476,96.0,0.513235,0.345202,0.943797,False,False,borderline_0.50-0.74
4,MR_0005,brain_results.csv,brain_mri,176,BRATS_373_z114,BRATS_373,114.0,0.514322,0.346187,0.893467,False,False,borderline_0.50-0.74
...,...,...,...,...,...,...,...,...,...,...,...,...,...
215,MR_0216,kits23_tumor_sam_boxprompt_metrics.csv,kidney_ct,1530,case_00028_z115.png,case_00028,115.0,0.926377,0.862851,NaN,False,False,strong_>=0.75
216,MR_0217,kits23_tumor_sam_boxprompt_metrics.csv,kidney_ct,748,case_00013_z142.png,case_00013,142.0,0.942788,0.891768,NaN,False,False,strong_>=0.75
217,MR_0218,kits23_tumor_sam_boxprompt_metrics.csv,kidney_ct,88,case_00002_z330.png,case_00002,330.0,0.948673,0.902357,NaN,False,False,strong_>=0.75
218,MR_0219,kits23_tumor_sam_boxprompt_metrics.csv,kidney_ct,1354,case_00025_z413.png,case_00025,413.0,0.978227,0.957382,NaN,False,False,strong_>=0.75


Candidate benchmark rows: 220

By domain / stratum:


,domain,sampling_stratum,n
0,brain_mri,borderline_0.50-0.74,25
1,brain_mri,failure_0.10-0.49,25
2,brain_mri,severe_<0.10,19
3,brain_mri,strong_>=0.75,25
4,histopathology,borderline_0.50-0.74,11
5,histopathology,failure_0.10-0.49,25
6,kidney_ct,borderline_0.50-0.74,25
7,kidney_ct,failure_0.10-0.49,25
8,kidney_ct,severe_<0.10,15
9,kidney_ct,strong_>=0.75,25


Saved: /content/drive/MyDrive/MedReasoner_SAM_2026/01_manifests/candidate_medreasoner_benchmark.csv


In [18]:
# CELL 11 — Discover image assets that are ALREADY available
# We only index the GitHub repo and Drive paths found by the targeted search.
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}
MED_EXTS = {".nii", ".gz", ".npy", ".npz"}

asset_rows = []

# 1) GitHub repo
for p in REPO_DIR.rglob("*"):
    if p.is_file() and (p.suffix.lower() in IMAGE_EXTS or p.suffix.lower() in MED_EXTS):
        asset_rows.append({
            "source": "github_clone",
            "path": str(p),
            "name": p.name,
            "stem": p.stem,
            "suffix": p.suffix.lower()
        })

# 2) Candidate Drive locations only
candidate_roots = set()
if len(drive_hits_df):
    for x in drive_hits_df["path"].tolist():
        p = Path(x)
        if p.is_dir():
            candidate_roots.add(p)
        elif p.exists():
            candidate_roots.add(p.parent)

# Avoid nested duplicate scans.
roots_sorted = sorted(candidate_roots, key=lambda p: len(str(p)))
pruned_roots = []
for p in roots_sorted:
    if not any(str(p).startswith(str(q) + os.sep) or p == q for q in pruned_roots):
        pruned_roots.append(p)

MAX_FILES_PER_ROOT = 50000
for root in pruned_roots[:30]:
    count = 0
    try:
        for p in root.rglob("*"):
            if p.is_file():
                count += 1
                if p.suffix.lower() in IMAGE_EXTS or p.suffix.lower() in MED_EXTS:
                    asset_rows.append({
                        "source": "drive",
                        "path": str(p),
                        "name": p.name,
                        "stem": p.stem,
                        "suffix": p.suffix.lower()
                    })
            if count >= MAX_FILES_PER_ROOT:
                print("Stopped large scan at", root, "after", count, "files.")
                break
    except Exception as e:
        print("Could not scan", root, ":", e)

assets = pd.DataFrame(asset_rows).drop_duplicates(subset=["path"]) if asset_rows else pd.DataFrame(
    columns=["source","path","name","stem","suffix"]
)

print("Discovered candidate image/medical files:", len(assets))
display(assets.head(100))

assets_path = DIRS["audit"] / "discovered_existing_assets.csv"
assets.to_csv(assets_path, index=False)
print("Saved:", assets_path)

Discovered candidate image/medical files: 435


,source,path,name,stem,suffix
0,github_clone,/content/sam-brats-robustness-audit/case_00000...,case_00000_z186.png,case_00000_z186,.png
1,github_clone,/content/sam-brats-robustness-audit/case_00001...,case_00001_z139.png,case_00001_z139,.png
2,github_clone,/content/sam-brats-robustness-audit/train_009_...,train_009_tissue2_TCGA-HE-7130-01Z-00-DX1.png,train_009_tissue2_TCGA-HE-7130-01Z-00-DX1,.png
3,github_clone,/content/sam-brats-robustness-audit/case_00000...,case_00000_z172.png,case_00000_z172,.png
4,github_clone,/content/sam-brats-robustness-audit/case_00000...,case_00000_z187.png,case_00000_z187,.png
...,...,...,...,...,...
95,github_clone,/content/sam-brats-robustness-audit/case_00001...,case_00001_z114.png,case_00001_z114,.png
96,github_clone,/content/sam-brats-robustness-audit/train_031_...,train_031_tissue4_TCGA-G9-6362-01Z-00-DX1.png,train_031_tissue4_TCGA-G9-6362-01Z-00-DX1,.png
97,github_clone,/content/sam-brats-robustness-audit/case_00000...,case_00000_z168.png,case_00000_z168,.png
98,github_clone,/content/sam-brats-robustness-audit/test_008_t...,test_008_tissue0_TCGA-AO-A0J2-01A-01-BSA.png,test_008_tissue0_TCGA-AO-A0J2-01A-01-BSA,.png


Saved: /content/drive/MyDrive/MedReasoner_SAM_2026/00_setup_audit/discovered_existing_assets.csv


In [19]:
# CELL 12 — Attempt conservative record-ID ↔ image matching
# Exact/near-exact filename matching only. We do NOT guess ambiguous files.
def normalize_key(x):
    x = str(x).lower()
    x = re.sub(r"\.(nii\.gz|nii|png|jpg|jpeg|tif|tiff|bmp|npy|npz)$", "", x)
    return re.sub(r"[^a-z0-9]+", "_", x).strip("_")

if len(assets):
    assets = assets.copy()
    assets["norm"] = assets["name"].map(normalize_key)

asset_index = {}
if len(assets):
    for _, r in assets.iterrows():
        asset_index.setdefault(r["norm"], []).append(r["path"])

candidate_with_assets = candidate.copy()

matched_paths = []
match_counts = []

if len(candidate_with_assets):
    for _, r in candidate_with_assets.iterrows():
        keys = {
            normalize_key(r.get("record_id","")),
            normalize_key(r.get("case_id",""))
        }
        z = r.get("z", np.nan)
        if pd.notna(z):
            try:
                z_int = int(float(z))
                case = normalize_key(r.get("case_id",""))
                keys.update({
                    normalize_key(f"{case}_z{z_int}"),
                    normalize_key(f"{case}_z{z_int:03d}")
                })
            except Exception:
                pass

        matches = []
        for k in keys:
            matches.extend(asset_index.get(k, []))

        matches = sorted(set(matches))
        matched_paths.append(matches[0] if len(matches) == 1 else "")
        match_counts.append(len(matches))

    candidate_with_assets["matched_existing_asset"] = matched_paths
    candidate_with_assets["asset_match_count"] = match_counts
    candidate_with_assets["asset_status"] = np.select(
        [
            candidate_with_assets["asset_match_count"] == 1,
            candidate_with_assets["asset_match_count"] > 1
        ],
        [
            "exact_or_near_exact_match",
            "ambiguous_multiple_matches"
        ],
        default="not_found_by_filename"
    )

display(candidate_with_assets.head(100))

matched_path = DIRS["manifests"] / "candidate_benchmark_with_asset_audit.csv"
candidate_with_assets.to_csv(matched_path, index=False)
print("Saved:", matched_path)

if len(candidate_with_assets):
    print("\nAsset status:")
    display(candidate_with_assets.groupby(["domain","asset_status"]).size().reset_index(name="n"))

,benchmark_id,source_csv,domain,row_index,record_id,case_id,z,dice,iou,sam_score,sam_failure_primary,sam_failure_severe,sampling_stratum,matched_existing_asset,asset_match_count,asset_status
0,MR_0001,brain_results.csv,brain_mri,45,BRATS_371_z075,BRATS_371,75.0,0.506769,0.339378,0.954224,False,False,borderline_0.50-0.74,,0,not_found_by_filename
1,MR_0002,brain_results.csv,brain_mri,297,BRATS_010_z091,BRATS_010,91.0,0.507427,0.339968,0.952594,False,False,borderline_0.50-0.74,,0,not_found_by_filename
2,MR_0003,brain_results.csv,brain_mri,196,BRATS_152_z091,BRATS_152,91.0,0.508024,0.340504,0.978911,False,False,borderline_0.50-0.74,,0,not_found_by_filename
3,MR_0004,brain_results.csv,brain_mri,137,BRATS_476_z096,BRATS_476,96.0,0.513235,0.345202,0.943797,False,False,borderline_0.50-0.74,,0,not_found_by_filename
4,MR_0005,brain_results.csv,brain_mri,176,BRATS_373_z114,BRATS_373,114.0,0.514322,0.346187,0.893467,False,False,borderline_0.50-0.74,,0,not_found_by_filename
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,MR_0096,monuseg_sam_boxprompt_metrics.csv,histopathology,26,train_012_tissue0_TCGA-MH-A561-01Z-00-DX1.png,train_012_tissue0_TCGA-MH-A561-01Z-00-DX1.png,NaN,0.517583,0.349148,NaN,False,False,borderline_0.50-0.74,/content/sam-brats-robustness-audit/train_012_...,1,exact_or_near_exact_match
96,MR_0097,monuseg_sam_boxprompt_metrics.csv,histopathology,32,train_018_tissue3_TCGA-49-4488-01Z-00-DX1.png,train_018_tissue3_TCGA-49-4488-01Z-00-DX1.png,NaN,0.524525,0.355495,NaN,False,False,borderline_0.50-0.74,/content/sam-brats-robustness-audit/train_018_...,1,exact_or_near_exact_match
97,MR_0098,monuseg_sam_boxprompt_metrics.csv,histopathology,43,train_029_tissue7_TCGA-RD-A8N9-01A-01-TS1.png,train_029_tissue7_TCGA-RD-A8N9-01A-01-TS1.png,NaN,0.530008,0.360552,NaN,False,False,borderline_0.50-0.74,/content/sam-brats-robustness-audit/train_029_...,1,exact_or_near_exact_match
98,MR_0099,monuseg_sam_boxprompt_metrics.csv,histopathology,39,train_025_tissue0_TCGA-F9-A8NY-01Z-00-DX1.png,train_025_tissue0_TCGA-F9-A8NY-01Z-00-DX1.png,NaN,0.549891,0.379207,NaN,False,False,borderline_0.50-0.74,/content/sam-brats-robustness-audit/train_025_...,1,exact_or_near_exact_match


Saved: /content/drive/MyDrive/MedReasoner_SAM_2026/01_manifests/candidate_benchmark_with_asset_audit.csv

Asset status:


,domain,asset_status,n
0,brain_mri,not_found_by_filename,94
1,histopathology,exact_or_near_exact_match,36
2,kidney_ct,exact_or_near_exact_match,8
3,kidney_ct,not_found_by_filename,82


In [20]:
# CELL 13 — Final readiness report
status = {
    "project_root": str(PROJECT_ROOT),
    "github_repo": str(REPO_DIR),
    "drive_mounted": bool(MYDRIVE.exists()),
    "drive_search_hits": int(len(drive_hits_df)),
    "old_notebook_path_hints": int(len(path_hints_df)),
    "github_csv_tables": int(len(csv_inventory_df)),
    "per_case_metric_rows_recovered": int(len(master)),
    "candidate_benchmark_rows": int(len(candidate)),
    "candidate_assets_discovered": int(len(assets)),
    "domains_in_master": sorted(master["domain"].dropna().unique().tolist()) if len(master) else [],
    "target_domains": TARGET_DOMAINS,
    "primary_failure_definition": "Dice < 0.50",
    "severe_failure_definition": "Dice < 0.10",
    "next_step": (
        "Use this audit to decide which domains already have image/mask assets. "
        "Only regenerate missing case assets; then run VLM failure-recognition experiments."
    )
}

if len(candidate_with_assets):
    status["benchmark_rows_with_single_asset_match"] = int(
        (candidate_with_assets["asset_match_count"] == 1).sum()
    )
    status["benchmark_rows_without_filename_match"] = int(
        (candidate_with_assets["asset_match_count"] == 0).sum()
    )

status_path = DIRS["audit"] / "setup_status.json"
status_path.write_text(json.dumps(status, indent=2))

print(json.dumps(status, indent=2))
print("\nSaved:", status_path)

print("\n" + "="*80)
print("WHAT TO SEND BACK")
print("="*80)
print("After the notebook finishes, send me either:")
print("1) a screenshot / copy of the CELL 13 JSON output, OR")
print("2) the file: 00_setup_audit/setup_status.json")
print("\nThen the next notebook can be generated against the data that actually exists.")

{
  "project_root": "/content/drive/MyDrive/MedReasoner_SAM_2026",
  "github_repo": "/content/sam-brats-robustness-audit",
  "drive_mounted": true,
  "drive_search_hits": 36,
  "old_notebook_path_hints": 632,
  "github_csv_tables": 15,
  "per_case_metric_rows_recovered": 2338,
  "candidate_benchmark_rows": 220,
  "candidate_assets_discovered": 435,
  "domains_in_master": [
    "brain_mri",
    "histopathology",
    "kidney_ct"
  ],
  "target_domains": [
    "brain_mri",
    "kidney_ct",
    "histopathology"
  ],
  "primary_failure_definition": "Dice < 0.50",
  "severe_failure_definition": "Dice < 0.10",
  "next_step": "Use this audit to decide which domains already have image/mask assets. Only regenerate missing case assets; then run VLM failure-recognition experiments.",
  "benchmark_rows_with_single_asset_match": 44,
  "benchmark_rows_without_filename_match": 176
}

Saved: /content/drive/MyDrive/MedReasoner_SAM_2026/00_setup_audit/setup_status.json

WHAT TO SEND BACK
After the notebo